## Condemned and Dead-End Houses Data Cleaning

In [5]:
import pandas as pd

output_columns = [
    "_id",
    "parcel_id",
    "address",
    "zip_code",
    "owner",
    "property_type",
    "create_date",
    "latest_inspection_result",
    "latest_inspection_score",
    "record_number",
    "inspection_status",
    "latitude",
    "longitude",
    "neighborhood",
]

df = pd.read_csv("../data/raw.csv", dtype={"parcel_id": str})

df = (df
    .replace("", pd.NA)
    .dropna(how="all")
    .assign(
        create_date=lambda x: pd.to_datetime(x["create_date"], errors="coerce"),
        _id_num=lambda x: pd.to_numeric(x["_id"], errors="coerce"),
    )
)

# Keep the most recent record with a valid inspection score for each parcel.
df = (df[df["latest_inspection_score"].notna()]
    .sort_values(["parcel_id", "create_date", "_id_num"], ascending=[True, False, False], na_position="last")
    .drop_duplicates(subset="parcel_id", keep="first")
    .drop(columns="_id_num")
    .loc[:, output_columns]
    .reset_index(drop=True)
)

df.to_csv("../data/cleaned_data.csv", index=False)

df.head()

,_id,parcel_id,address,zip_code,owner,property_type,create_date,latest_inspection_result,latest_inspection_score,record_number,inspection_status,latitude,longitude,neighborhood
0,53193,0004J00093000000,"511 GRACE ST, Pittsburgh, 15211",15211,CHESNEY VICTOR,Condemned/Dead End Property,2026-02-24,Fail,2.0,0004J00093000000,Active,40.427750,-80.014599,Mount Washington
1,54465,0019G00104000000,"507 HARKER ST, Pittsburgh, 15220",15220,BOGDAN JOSEPH J & MARGARET Z,Condemned/Dead End Property,2026-02-23,NaN,NaN,0019G00104000000,Active,40.439264,-80.037590,Elliott
2,53584,0044H00278000000,"2825 CALIFORNIA AVE, Pittsburgh, 15212",15212,NaN,Condemned/Dead End Property,2026-02-23,Fail,1.0,0044H00278000000,Active,40.469492,-80.036066,Marshall-Shadeland
3,53824,0045M00403000000,"544 W BURGESS ST, Pittsburgh, 15214",15214,INTISSAR LLC,Condemned/Dead End Property,2026-02-23,NaN,NaN,0045M00403000000,Active,40.465341,-80.016433,Perry South
4,54023,0047E00196000000,"86 LUELLA ST, Pittsburgh, 15212",15212,FOX JORDAN,Condemned/Dead End Property,2026-02-23,NaN,NaN,0047E00196000000,Active,40.468630,-79.997816,Spring Hill-City View


In [8]:
# Calculate the proportion of "No primary address specified"
no_address_count = (df['address'] == "No primary address specified").sum()
total_count = len(df)
percentage = (no_address_count / total_count) * 100

print(f"Total records: {total_count}")
print(f"Number of 'No primary address specified': {no_address_count}")
print(f"Proportion: {percentage:.2f}%")

Total records: 2665
Number of 'No primary address specified': 492
Proportion: 18.46%
